# CS535 Computational Biology — English Edition
## CNN Research Lab: Cats vs. Dogs → Modern Deep Learning → Pretrained Models

**Environment:** Google Colab / Python / PyTorch 2.x / GPU recommended  
**Dataset scale:** Microsoft Cats vs. Dogs, ~25,000 raw images (after cleaning, ~23K usable images is typical)

### Learning objectives
1. Understand CNN operations, architecture, parameters, receptive fields, and model capacity.
2. Perform **controlled experiments** on depth, kernel size, filters, pooling, dropout, input resolution, optimizer, learning rate, batch size, and augmentation.
3. Compare modern deep learning models: ResNet, MobileNet, EfficientNet, ConvNeXt.
4. Compare **from-scratch vs. pretrained feature extraction vs. fine-tuning**.
5. Evaluate models with Accuracy, Balanced Accuracy, Precision, Recall/Sensitivity, Specificity, F1, ROC-AUC, PR-AUC, MCC, Brier score, TP/TN/FP/FN.
6. Build and evaluate an **independent hard / OOD challenge set**.

> Research rule: validation data may be used for model/hyperparameter selection. The independent test set must not be repeatedly used to tune the model.

## 0. Runtime setup

In [ ]:
# Colab: Runtime -> Change runtime type -> T4 GPU (recommended)
import os, sys, math, random, time, json, shutil, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import transforms, models
from PIL import Image, ImageOps, UnidentifiedImageError

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, matthews_corrcoef, brier_score_loss,
    roc_curve, precision_recall_curve
)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SEED = 535
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Reproducibility note:
# Strict deterministic mode can reduce performance and not every CUDA operation is deterministic.
torch.backends.cudnn.benchmark = True

## 1. Global experiment configuration

`FAST_MODE=True` is suitable for classroom demonstration.  
For a formal report, set `FAST_MODE=False` and increase epochs.

### Resolution policy
For the controlled CNN resolution experiments, this notebook uses **no-upscale + pad**:
- If an original image is larger than the target, it can be downscaled.
- If it is smaller, it is **not enlarged**; it is padded to the target canvas.

This avoids creating artificial detail through upscaling.

For pretrained models, official TorchVision weight transforms are normally preferred because preprocessing is part of the pretrained recipe. This notebook provides a consistent custom protocol for fair course experiments; students should discuss this distinction in the report.

In [ ]:
FAST_MODE = True

IMG_SIZE = 128 if FAST_MODE else 224
BATCH_SIZE = 64 if torch.cuda.is_available() else 32
NUM_WORKERS = 2
BASE_EPOCHS = 3 if FAST_MODE else 12
PRETRAIN_EPOCHS = 2 if FAST_MODE else 6

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

DATA_ROOT = Path("/content/cs535_catsdogs")
RAW_ZIP = DATA_ROOT / "kagglecatsanddogs_5340.zip"
EXTRACT_DIR = DATA_ROOT / "raw"
PET_DIR = EXTRACT_DIR / "PetImages"
RESULTS_DIR = Path("/content/cs535_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print({
    "FAST_MODE": FAST_MODE,
    "IMG_SIZE": IMG_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "BASE_EPOCHS": BASE_EPOCHS,
    "PRETRAIN_EPOCHS": PRETRAIN_EPOCHS,
})

# Warm-up bridge — MNIST before Cats vs. Dogs

MNIST is used here as a **conceptual bridge**:

```text
5×5 hand calculation
      ↓
MNIST 28×28 grayscale
      ↓
Cats vs. Dogs natural images
```

With MNIST, the image is simple enough that we can inspect:
- actual pixels,
- handcrafted edge filters,
- multiple feature maps,
- learned convolution kernels,
- intermediate CNN activations.

Standard MNIST contains **60,000 training images and 10,000 test images**, each 28×28 grayscale, with 10 classes (digits 0–9).

In [ ]:
from torchvision.datasets import MNIST

MNIST_ROOT = Path("/content/mnist_data")

mnist_train = MNIST(
    root=MNIST_ROOT, train=True, download=True,
    transform=transforms.ToTensor()
)
mnist_test = MNIST(
    root=MNIST_ROOT, train=False, download=True,
    transform=transforms.ToTensor()
)

print("MNIST train:", len(mnist_train))
print("MNIST test :", len(mnist_test))
print("One image shape:", mnist_train[0][0].shape)
print("One label:", mnist_train[0][1])

## MNIST 1 — Look at actual 28×28 images and pixels

In [ ]:
fig = plt.figure(figsize=(10,4))
for i in range(10):
    x, y = mnist_train[i]
    ax = plt.subplot(2,5,i+1)
    ax.imshow(x.squeeze(0), cmap="gray")
    ax.set_title(f"label={y}")
    ax.axis("off")
plt.tight_layout()
plt.show()

x0, y0 = mnist_train[0]
display(pd.DataFrame((x0.squeeze(0).numpy()*255).round().astype(int)).iloc[8:20, 8:20])

## MNIST 2 — Apply multiple handcrafted kernels

This is the same idea as the manual convolution demo, but now on a real MNIST image.

We will apply:
- vertical edge,
- horizontal edge,
- diagonal/corner-like,
- blur/local-density.

One input image + four kernels → four feature maps.

In [ ]:
hand_kernels = {
    "vertical_edge": torch.tensor([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=torch.float32),
    "horizontal_edge": torch.tensor([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=torch.float32),
    "diagonal": torch.tensor([[-1,-1,0],[-1,0,1],[0,1,1]], dtype=torch.float32),
    "blur": torch.ones((3,3), dtype=torch.float32)/9.0,
}

sample_img, sample_label = mnist_train[0]
batch = sample_img.unsqueeze(0)  # [1,1,28,28]

fig = plt.figure(figsize=(12,3))
ax = plt.subplot(1,5,1)
ax.imshow(sample_img.squeeze(), cmap="gray")
ax.set_title(f"Input: {sample_label}")
ax.axis("off")

for idx,(name,k) in enumerate(hand_kernels.items(), start=2):
    weight = k.view(1,1,3,3)
    feat = F.conv2d(batch, weight, padding=1)
    ax = plt.subplot(1,5,idx)
    ax.imshow(feat.squeeze().numpy(), cmap="gray")
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## MNIST 3 — A small CNN with multiple learned filters

Architecture:

```text
28×28×1
  ↓ Conv 3×3, 8 filters
28×28×8
  ↓ ReLU + MaxPool
14×14×8
  ↓ Conv 3×3, 16 filters
14×14×16
  ↓ ReLU + MaxPool
7×7×16
  ↓ Flatten
10 classes
```

The first layer has **8 different learned kernels**.  
Therefore it produces **8 feature maps** for every input image.

In [ ]:
class SmallMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16*7*7, 10)

    def forward(self, x, return_features=False):
        f1 = F.relu(self.conv1(x))      # [B,8,28,28]
        p1 = self.pool(f1)              # [B,8,14,14]
        f2 = F.relu(self.conv2(p1))     # [B,16,14,14]
        p2 = self.pool(f2)              # [B,16,7,7]
        logits = self.fc(p2.flatten(1))
        if return_features:
            return logits, f1, f2
        return logits

mnist_model = SmallMNISTCNN().to(DEVICE)
print(mnist_model)
print("Parameters:", count_params(mnist_model))

In [ ]:
mnist_train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True, num_workers=2)
mnist_test_loader = DataLoader(mnist_test, batch_size=256, shuffle=False, num_workers=2)

optimizer = torch.optim.Adam(mnist_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
MNIST_EPOCHS = 1 if FAST_MODE else 3

for epoch in range(MNIST_EPOCHS):
    mnist_model.train()
    running_loss = 0.0
    correct = total = 0
    for xb, yb in mnist_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = mnist_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*yb.size(0)
        correct += (logits.argmax(1)==yb).sum().item()
        total += yb.size(0)
    print(f"epoch={epoch+1}, loss={running_loss/total:.4f}, train_acc={correct/total:.4f}")

mnist_model.eval()
correct=total=0
with torch.no_grad():
    for xb,yb in mnist_test_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        pred=mnist_model(xb).argmax(1)
        correct+=(pred==yb).sum().item()
        total+=yb.size(0)
print("MNIST test accuracy:", correct/total)

## MNIST 4 — Visualize learned first-layer kernels

In [ ]:
w = mnist_model.conv1.weight.detach().cpu()  # [8,1,3,3]

fig = plt.figure(figsize=(12,3))
for i in range(w.shape[0]):
    ax = plt.subplot(2,4,i+1)
    ax.imshow(w[i,0], cmap="gray")
    ax.set_title(f"kernel {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("First-layer weight shape:", tuple(w.shape))
print("Interpretation: 8 learned kernels, each 3×3×1")

## MNIST 5 — Visualize the 8 feature maps from one digit

This is the key connection:

> One image passes through 8 learned kernels → 8 different activation maps.

Some maps may respond more strongly to vertical strokes, curves, corners, or specific local combinations.

In [ ]:
mnist_model.eval()
sample_img, sample_label = mnist_test[0]
with torch.no_grad():
    logits, f1, f2 = mnist_model(sample_img.unsqueeze(0).to(DEVICE), return_features=True)

f1 = f1.squeeze(0).cpu()

fig = plt.figure(figsize=(12,6))
ax = plt.subplot(3,3,1)
ax.imshow(sample_img.squeeze(), cmap="gray")
ax.set_title(f"input label={sample_label}")
ax.axis("off")

for i in range(8):
    ax = plt.subplot(3,3,i+2)
    ax.imshow(f1[i], cmap="gray")
    ax.set_title(f"feature map {i}")
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Predicted class:", int(logits.argmax(1).item()))

## MNIST 6 — Bridge to Cats vs. Dogs

MNIST teaches the mechanics cleanly:

```text
one grayscale image
  ↓
many learned filters
  ↓
many feature maps
  ↓
deeper filters combine earlier feature maps
  ↓
digit classification
```

Cats vs. Dogs uses exactly the same logic, but:
- RGB instead of grayscale,
- much larger images,
- more complex textures/backgrounds,
- more filters and deeper networks,
- stronger need for augmentation and transfer learning,
- greater risk of overfitting and distribution shift.

Now we move to the large natural-image experiment.

## 2. Download the large Cats vs. Dogs dataset

The original Microsoft archive is about **786 MB** and contains roughly **25,000 raw cat/dog images**.  
Real-world datasets contain corrupted files, so the next step explicitly validates images.

If the Microsoft URL changes in the future, use the TFDS `cats_vs_dogs` dataset or Kaggle Dogs vs. Cats as an alternative source.

In [ ]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = (
    "https://download.microsoft.com/download/3/E/1/"
    "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
)

import urllib.request

if not RAW_ZIP.exists():
    print("Downloading ~786 MB dataset...")
    urllib.request.urlretrieve(DATA_URL, RAW_ZIP)
else:
    print("Zip already exists:", RAW_ZIP)

if not PET_DIR.exists():
    print("Extracting...")
    with zipfile.ZipFile(RAW_ZIP, "r") as zf:
        zf.extractall(EXTRACT_DIR)

print("Dataset directory:", PET_DIR)
print("Cat files:", len(list((PET_DIR / "Cat").glob("*"))))
print("Dog files:", len(list((PET_DIR / "Dog").glob("*"))))

## 3. Data quality control and manifest

In [ ]:
def build_manifest(pet_dir: Path):
    rows = []
    bad = []
    for label_name, label in [("Cat", 0), ("Dog", 1)]:
        folder = pet_dir / label_name
        for p in folder.iterdir():
            if not p.is_file():
                continue
            try:
                with Image.open(p) as im:
                    im.verify()
                with Image.open(p) as im:
                    w, h = im.size
                    mode = im.mode
                rows.append({
                    "path": str(p),
                    "label": label,
                    "class_name": label_name.lower(),
                    "width": w,
                    "height": h,
                    "min_side": min(w, h),
                    "max_side": max(w, h),
                    "aspect_ratio": w / h if h else np.nan,
                    "mode": mode,
                })
            except Exception as e:
                bad.append((str(p), type(e).__name__))
    return pd.DataFrame(rows), pd.DataFrame(bad, columns=["path", "error"])

manifest, bad_files = build_manifest(PET_DIR)
print("Valid images:", len(manifest))
print("Invalid/corrupted:", len(bad_files))
display(manifest.head())
display(manifest.groupby("class_name").size().rename("count"))

manifest.to_csv(RESULTS_DIR / "dataset_manifest.csv", index=False)
bad_files.to_csv(RESULTS_DIR / "bad_files.csv", index=False)

In [ ]:
# Inspect image-size distribution
fig = plt.figure(figsize=(8,5))
plt.hist(manifest["min_side"], bins=50)
plt.xlabel("Shorter side (pixels)")
plt.ylabel("Number of images")
plt.title("Original image size distribution")
plt.show()

display(manifest[["width","height","min_side","max_side","aspect_ratio"]].describe().round(2))

## 4. Stratified train / validation / independent test split

In [ ]:
train_df, temp_df = train_test_split(
    manifest, test_size=(1 - TRAIN_FRAC),
    stratify=manifest["label"], random_state=SEED
)

relative_test = TEST_FRAC / (VAL_FRAC + TEST_FRAC)
val_df, test_df = train_test_split(
    temp_df, test_size=relative_test,
    stratify=temp_df["label"], random_state=SEED
)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(df), df["label"].value_counts(normalize=True).sort_index().round(4).to_dict())
    df.to_csv(RESULTS_DIR / f"{name}_split.csv", index=False)

## 5. Visualize random samples

Before modeling, always look at the data. Ask:
- Are labels plausible?
- Are there unusual backgrounds?
- Are some images tiny?
- Could there be duplicates or leakage?

In [ ]:
def show_samples(df, n=12):
    sample = df.sample(n=min(n, len(df)), random_state=SEED)
    cols = 4
    rows = math.ceil(len(sample)/cols)
    plt.figure(figsize=(12, 3*rows))
    for i, (_, r) in enumerate(sample.iterrows(), 1):
        with Image.open(r["path"]) as im:
            im = im.convert("RGB")
            plt.subplot(rows, cols, i)
            plt.imshow(im)
            plt.axis("off")
            plt.title(f'{r["class_name"]} | {r["width"]}×{r["height"]}')
    plt.tight_layout()
    plt.show()

show_samples(train_df, 12)

## 6. Transform: no-upscale + padding

This transform:
1. preserves aspect ratio,
2. downsizes only when needed,
3. never enlarges a smaller image,
4. centers the image on a fixed square canvas.

For augmentation we add mild horizontal flip / rotation / color jitter.  
Students should justify whether each augmentation preserves class semantics.

In [ ]:
class ResizeNoUpscalePad:
    def __init__(self, size, fill=(0,0,0)):
        self.size = int(size)
        self.fill = fill

    def __call__(self, img):
        img = img.convert("RGB")
        w, h = img.size
        scale = min(self.size / w, self.size / h, 1.0)  # never > 1
        new_w, new_h = max(1, int(round(w*scale))), max(1, int(round(h*scale)))
        if (new_w, new_h) != (w, h):
            img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        left = (self.size - new_w)//2
        top = (self.size - new_h)//2
        canvas.paste(img, (left, top))
        return canvas

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def make_transform(size=224, train=False, augment=False, normalize=True):
    ops = [ResizeNoUpscalePad(size)]
    if train and augment:
        ops += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
        ]
    ops += [transforms.ToTensor()]
    if normalize:
        ops += [transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

## 7. Dataset and DataLoader

In [ ]:
class CatsDogsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["path"]) as im:
            image = im.convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(float(row["label"]), dtype=torch.float32)
        return image, label, row["path"]

def make_loaders(size=IMG_SIZE, batch_size=BATCH_SIZE, augment=True):
    train_ds = CatsDogsDataset(train_df, make_transform(size, train=True, augment=augment))
    val_ds   = CatsDogsDataset(val_df,   make_transform(size, train=False))
    test_ds  = CatsDogsDataset(test_df,  make_transform(size, train=False))
    common = dict(batch_size=batch_size, num_workers=NUM_WORKERS,
                  pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, shuffle=True, **common)
    val_loader   = DataLoader(val_ds, shuffle=False, **common)
    test_loader  = DataLoader(test_ds, shuffle=False, **common)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders()
x, y, paths = next(iter(train_loader))
print(x.shape, y.shape, y[:8])

# CNN conceptual framework — Hung-yi Lee + CS231n

### Observation 1 — Local patterns
Useful visual patterns usually occupy only part of an image → **local receptive field**.

### Observation 2 — Same pattern, different position
Reuse the same weights across spatial locations → **parameter sharing / convolution**.

### Observation 3 — Moderate subsampling
Spatial downsampling can preserve object identity → **pooling/downsampling**, but pooling is not mandatory for every task.

Key terms: receptive field, kernel/filter, channel, feature map, stride, padding, parameter sharing, pooling, hierarchical representation.

## Convolution output size

\[
O=\left\lfloor\frac{I+P_{left}+P_{right}-D(K-1)-1}{S}+1\right\rfloor
\]

Odd kernels such as 3,5,7 can use symmetric SAME padding.  
Even kernels 2,4,6,10 require **asymmetric SAME padding** to preserve exact size.

# Part A — CNN fundamentals

## 8. A configurable CNN for controlled experiments

Parameters to study:
- `depth`
- `base_filters`
- `kernel_size`
- `pool`
- `dropout`
- input resolution

The network uses `AdaptiveAvgPool2d(1)` so the final classifier does not depend on a fixed feature-map size.

In [ ]:
class SamePadConv2d(nn.Module):
    # TensorFlow-style SAME padding for stride=1; supports even kernels.
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, dilation=1, bias=True):
        super().__init__()
        kh = kw = kernel_size if isinstance(kernel_size, int) else None
        if not isinstance(kernel_size, int):
            kh, kw = kernel_size
        dh = dw = dilation if isinstance(dilation, int) else None
        if not isinstance(dilation, int):
            dh, dw = dilation
        self.kh, self.kw = kh, kw
        self.dh, self.dw = dh, dw
        self.stride = stride
        self.conv = nn.Conv2d(in_channels, out_channels, (kh,kw),
                              stride=stride, padding=0,
                              dilation=(dh,dw), bias=bias)

    def forward(self, x):
        if self.stride != 1:
            return self.conv(x)
        total_h = self.dh * (self.kh - 1)
        total_w = self.dw * (self.kw - 1)
        top, left = total_h // 2, total_w // 2
        bottom, right = total_h - top, total_w - left
        return self.conv(F.pad(x, (left, right, top, bottom)))


class ConfigurableCNN(nn.Module):
    def __init__(self, depth=3, base_filters=32, kernel_size=3,
                 pool="max", pool_kernel=2, pool_stride=2,
                 dropout=0.3, batchnorm=True, dilation=1,
                 channel_growth=2, max_channels=512):
        super().__init__()
        layers, in_ch = [], 3
        for i in range(depth):
            out_ch = min(int(base_filters * (channel_growth ** i)), max_channels)
            layers.append(SamePadConv2d(in_ch, out_ch, kernel_size,
                                        stride=1, dilation=dilation))
            if batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            if pool == "max":
                layers.append(nn.MaxPool2d(pool_kernel, stride=pool_stride))
            elif pool == "avg":
                layers.append(nn.AvgPool2d(pool_kernel, stride=pool_stride))
            elif pool != "none":
                raise ValueError("pool must be max, avg, or none")
            in_ch = out_ch

        self.features = nn.Sequential(*layers)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(in_ch, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.classifier(x).squeeze(1)

model = ConfigurableCNN(depth=3, base_filters=32, kernel_size=3).to(DEVICE)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 9. CNN parameter-count exercise

For a convolution layer:

\[
\text{parameters} = (k_h k_w C_{in} + 1)\,C_{out}
\]

Example: RGB input, 3×3 kernel, 32 filters:

\[
(3\times3\times3+1)\times32=896
\]

Use the helper below to verify your manual calculation.

In [ ]:
def count_params(model, trainable_only=False):
    params = (p for p in model.parameters() if (p.requires_grad or not trainable_only))
    return sum(p.numel() for p in params)

print("All parameters:", count_params(model))
print("Trainable parameters:", count_params(model, trainable_only=True))

# Part B — Training and complete evaluation

In [ ]:
def run_epoch(model, loader, optimizer=None, criterion=None):
    is_train = optimizer is not None
    model.train(is_train)
    criterion = criterion or nn.BCEWithLogitsLoss()

    total_loss = 0.0
    ys, probs, all_paths = [], [], []

    for images, labels, paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        ys.extend(labels.detach().cpu().numpy().tolist())
        probs.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
        all_paths.extend(paths)

    return total_loss / len(loader.dataset), np.array(ys), np.array(probs), all_paths


def binary_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    out = {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc_avg_precision": average_precision_score(y_true, prob),
        "mcc": matthews_corrcoef(y_true, pred),
        "brier": brier_score_loss(y_true, prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "threshold": float(threshold),
    }
    return out


def fit_model(model, train_loader, val_loader, epochs=5, lr=1e-3,
              optimizer_name="adamw", weight_decay=1e-4):
    criterion = nn.BCEWithLogitsLoss()

    if optimizer_name.lower() == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                                    weight_decay=weight_decay)
    elif optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                     weight_decay=weight_decay)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                      weight_decay=weight_decay)

    history = []
    best_state = None
    best_val_auc = -np.inf

    for epoch in range(1, epochs+1):
        t0 = time.time()
        tr_loss, tr_y, tr_p, _ = run_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_y, va_p, _ = run_epoch(model, val_loader, None, criterion)
        tr_m = binary_metrics(tr_y, tr_p)
        va_m = binary_metrics(va_y, va_p)

        row = {
            "epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
            "train_acc": tr_m["accuracy"], "val_acc": va_m["accuracy"],
            "train_auc": tr_m["roc_auc"], "val_auc": va_m["roc_auc"],
            "seconds": time.time() - t0,
        }
        history.append(row)
        print(row)

        if va_m["roc_auc"] > best_val_auc:
            best_val_auc = va_m["roc_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return pd.DataFrame(history)

## 10. Train the baseline CNN

In [ ]:
train_loader, val_loader, test_loader = make_loaders(size=IMG_SIZE, augment=True)

baseline = ConfigurableCNN(
    depth=3, base_filters=32, kernel_size=3,
    pool="max", dropout=0.30, batchnorm=True
).to(DEVICE)

history = fit_model(
    baseline, train_loader, val_loader,
    epochs=BASE_EPOCHS, lr=1e-3,
    optimizer_name="adamw", weight_decay=1e-4
)

display(history)

In [ ]:
# Learning curves
fig = plt.figure(figsize=(8,5))
plt.plot(history["epoch"], history["train_loss"], marker="o", label="train")
plt.plot(history["epoch"], history["val_loss"], marker="o", label="validation")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

fig = plt.figure(figsize=(8,5))
plt.plot(history["epoch"], history["train_auc"], marker="o", label="train")
plt.plot(history["epoch"], history["val_auc"], marker="o", label="validation")
plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("Training vs Validation ROC-AUC")
plt.legend()
plt.show()

## 11. Choose a classification threshold on the validation set

The default threshold 0.5 is not sacred.  
Here we choose a threshold by maximizing validation F1 **without touching the test set**.

In [ ]:
_, val_y, val_prob, _ = run_epoch(baseline, val_loader)
thresholds = np.linspace(0.05, 0.95, 181)
scores = [f1_score(val_y, val_prob >= t) for t in thresholds]
best_threshold = float(thresholds[int(np.argmax(scores))])
print("Validation-selected threshold:", best_threshold)

## 12. Final independent test evaluation

In [ ]:
test_loss, test_y, test_prob, test_paths = run_epoch(baseline, test_loader)
test_metrics_05 = binary_metrics(test_y, test_prob, threshold=0.5)
test_metrics_opt = binary_metrics(test_y, test_prob, threshold=best_threshold)

print("Threshold = 0.5")
display(pd.DataFrame([test_metrics_05]).T.rename(columns={0:"value"}))
print("Validation-selected threshold")
display(pd.DataFrame([test_metrics_opt]).T.rename(columns={0:"value"}))

In [ ]:
# Confusion matrix
pred = (test_prob >= best_threshold).astype(int)
cm = confusion_matrix(test_y, pred, labels=[0,1])

fig = plt.figure(figsize=(5,5))
plt.imshow(cm)
plt.xticks([0,1], ["Cat","Dog"])
plt.yticks([0,1], ["Cat","Dog"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(test_y, test_prob)
fig = plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'AUC={test_metrics_opt["roc_auc"]:.3f}')
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Precision-Recall curve
precision, recall, _ = precision_recall_curve(test_y, test_prob)
fig = plt.figure(figsize=(6,5))
plt.plot(recall, precision, label=f'AP={test_metrics_opt["pr_auc_avg_precision"]:.3f}')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend()
plt.show()

# Part B2 — Fine-grained CNN parameter laboratory

Mandatory kernel sizes: **1, 2, 3, 4, 5, 6, 7, 10**  
Optional historical comparison: **11**

Mandatory filter counts: **32, 64, 96, 128, 256, 512, 1024**

Do not brute-force every combination. Start with one-factor-at-a-time controlled experiments.

In [ ]:
KERNEL_SIZES = [1,2,3,4,5,6,7,10]
FILTER_COUNTS = [32,64,96,128,256,512,1024]
POOL_TYPES = ["none","max","avg"]
POOL_KERNELS = [2,3,4,5]
POOL_STRIDES = [1,2,3]
DILATIONS = [1,2,3]
print(KERNEL_SIZES)
print(FILTER_COUNTS)

## Kernel interpretation
- 1×1: channel mixing / projection
- 2×2: even kernel, very local
- 3×3: classic small kernel
- 4×4: even-kernel alignment experiment
- 5×5: wider local context
- 6×6: larger even kernel
- 7×7: large receptive field
- 10×10: deliberate large-kernel stress test

In [ ]:
def estimate_conv_params(k, cin, cout, bias=True):
    return k*k*cin*cout + (cout if bias else 0)

kernel_cost = pd.DataFrame([
    {
        "kernel": f"{k}x{k}",
        "params_3_to_64": estimate_conv_params(k,3,64),
        "relative_to_3x3": estimate_conv_params(k,3,64)/estimate_conv_params(3,3,64)
    }
    for k in KERNEL_SIZES
])
display(kernel_cost)

In [ ]:
plt.figure(figsize=(10,6))
for i,k in enumerate(KERNEL_SIZES,1):
    ax=plt.subplot(2,4,i)
    canvas=np.zeros((12,12))
    start=(12-k)//2
    canvas[start:start+k,start:start+k]=1
    ax.imshow(canvas)
    ax.set_title(f"{k}x{k}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Width experiment — isolate filter count safely

If a network doubles channels every layer, starting from 1024 would become impractical.  
So the following network changes only the **first-layer number of filters**, then projects to a fixed 128-channel bottleneck.

In [ ]:
class WidthSweepCNN(nn.Module):
    def __init__(self, filters=64, kernel_size=3, dropout=0.3):
        super().__init__()
        self.conv1 = SamePadConv2d(3, filters, kernel_size)
        self.bn1 = nn.BatchNorm2d(filters)
        self.pool1 = nn.MaxPool2d(2)
        self.project = nn.Conv2d(filters, 128, kernel_size=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = SamePadConv2d(128,128,3)
        self.bn3 = nn.BatchNorm2d(128)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(128,1)

    def forward(self,x):
        x=self.pool1(F.relu(self.bn1(self.conv1(x))))
        x=self.pool2(F.relu(self.bn2(self.project(x))))
        x=F.relu(self.bn3(self.conv3(x)))
        x=self.gap(x).flatten(1)
        return self.fc(self.dropout(x)).squeeze(1)

for f in FILTER_COUNTS:
    m=WidthSweepCNN(f)
    print(f"{f:4d} filters -> {count_params(m)/1e6:.3f}M params")

In [ ]:
def recommended_batch_for_filters(filters):
    if filters >= 1024: return 4
    if filters >= 512: return 8
    if filters >= 256: return 16
    if filters >= 128: return 32
    return 64

for f in FILTER_COUNTS:
    print(f, recommended_batch_for_filters(f))

## Pooling experiment
Compare:
- none / max / average
- pool kernel 2 / 3 / 4 / 5
- stride 1 / 2 / 3
- global average pooling vs flatten+dense (extension)

In [ ]:
def pooling_output_size(n,k,s,p=0):
    return math.floor((n+2*p-k)/s+1)

display(pd.DataFrame([
    {"input":224,"pool_kernel":k,"stride":s,"output":pooling_output_size(224,k,s)}
    for k in POOL_KERNELS for s in POOL_STRIDES
]))

## Effective receptive field

Initialize \(r=1, j=1\).

\[
r_{new}=r_{old}+(K-1)j_{old}, \qquad j_{new}=j_{old}S
\]

Compare a single 7×7 convolution with three stacked 3×3 convolutions.

In [ ]:
def receptive_field(layers):
    r,j=1,1
    rows=[]
    for i,(k,s) in enumerate(layers,1):
        r=r+(k-1)*j
        j=j*s
        rows.append({"layer":i,"kernel":k,"stride":s,"receptive_field":r,"jump":j})
    return pd.DataFrame(rows)

display(receptive_field([(3,1),(3,1),(3,1)]))
display(receptive_field([(7,1)]))
display(receptive_field([(3,1),(2,2),(3,1),(2,2),(3,1)]))

# Part C — Controlled CNN experiments

## 13. Experiment runner

**Important:** use the same data split and change only the factor being studied.

`FAST_MODE` limits the amount of data/epochs only to make classroom demonstrations feasible.  
For your report, use the full split.

In [ ]:
def maybe_subset_loader(loader, max_items=4000):
    if not FAST_MODE:
        return loader
    ds = loader.dataset
    n = min(max_items, len(ds))
    idx = np.arange(n)
    sub = Subset(ds, idx)
    return DataLoader(
        sub, batch_size=loader.batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

def controlled_cnn_experiment(configs, size=IMG_SIZE, epochs=None):
    epochs = epochs or (2 if FAST_MODE else 8)
    tr, va, _ = make_loaders(size=size, augment=True)
    if FAST_MODE:
        tr = maybe_subset_loader(tr, 5000)

    rows = []
    for cfg in configs:
        print("\nCONFIG:", cfg)
        model = ConfigurableCNN(**cfg).to(DEVICE)
        hist = fit_model(model, tr, va, epochs=epochs, lr=1e-3)
        _, y, p, _ = run_epoch(model, va)
        m = binary_metrics(y, p)
        rows.append({
            **cfg,
            "params": count_params(model),
            "val_accuracy": m["accuracy"],
            "val_roc_auc": m["roc_auc"],
            "val_pr_auc": m["pr_auc_avg_precision"],
            "val_mcc": m["mcc"],
        })
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(rows)

### 13A. Kernel size experiment

In [ ]:
kernel_configs = [
    dict(depth=3, base_filters=32, kernel_size=k,
         pool="max", pool_kernel=2, pool_stride=2,
         dropout=0.3, batchnorm=True, dilation=1,
         channel_growth=2, max_channels=256)
    for k in [1,2,3,4,5,6,7,10]
]
kernel_results = controlled_cnn_experiment(kernel_configs)
display(kernel_results.sort_values("val_roc_auc", ascending=False))

### Filter-number sweep: 32 / 64 / 96 / 128 / 256 / 512 / 1024

In [ ]:
def controlled_width_experiment(filter_counts, size=None, epochs=None):
    size = size or (96 if FAST_MODE else 128)
    epochs = epochs or (1 if FAST_MODE else 5)
    rows=[]
    for f in filter_counts:
        bs=recommended_batch_for_filters(f)
        tr,va,_=make_loaders(size=size,batch_size=bs,augment=True)
        if FAST_MODE:
            tr=maybe_subset_loader(tr,2500)
        model=WidthSweepCNN(filters=f).to(DEVICE)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0=time.time()
        fit_model(model,tr,va,epochs=epochs,lr=1e-3)
        elapsed=time.time()-t0
        _,y,p,_=run_epoch(model,va)
        met=binary_metrics(y,p)
        peak=(torch.cuda.max_memory_allocated()/1024**2
              if torch.cuda.is_available() else np.nan)
        rows.append({
            "filters":f,"batch_size":bs,"params":count_params(model),
            "val_accuracy":met["accuracy"],"val_roc_auc":met["roc_auc"],
            "val_pr_auc":met["pr_auc_avg_precision"],"val_mcc":met["mcc"],
            "seconds":elapsed,"peak_gpu_mb":peak
        })
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(rows)

FILTERS_TO_RUN = [32,64,128,256] if FAST_MODE else FILTER_COUNTS
filter_results=controlled_width_experiment(FILTERS_TO_RUN)
display(filter_results)

In [ ]:
if len(filter_results):
    plt.figure(figsize=(8,5))
    plt.plot(filter_results["filters"],filter_results["val_roc_auc"],marker="o")
    plt.xscale("log",base=2)
    plt.xlabel("First-layer filters")
    plt.ylabel("Validation ROC-AUC")
    plt.title("Filter count vs ROC-AUC")
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(filter_results["filters"],filter_results["peak_gpu_mb"],marker="o")
    plt.xscale("log",base=2)
    plt.xlabel("First-layer filters")
    plt.ylabel("Peak GPU memory (MB)")
    plt.title("Filter count vs GPU memory")
    plt.show()

### Pooling type sweep

In [ ]:
pool_configs=[
    dict(depth=3,base_filters=32,kernel_size=3,pool=p,
         pool_kernel=2,pool_stride=2,dropout=0.3,batchnorm=True,
         dilation=1,channel_growth=2,max_channels=256)
    for p in ["none","max","avg"]
]
pool_type_results=controlled_cnn_experiment(pool_configs)
display(pool_type_results)

### Pooling window sweep

In [ ]:
pool_window_configs=[
    dict(depth=3,base_filters=32,kernel_size=3,pool="max",
         pool_kernel=k,pool_stride=2,dropout=0.3,batchnorm=True,
         dilation=1,channel_growth=2,max_channels=256)
    for k in [2,3,4,5]
]
pool_window_results=controlled_cnn_experiment(pool_window_configs)
display(pool_window_results)

### Dilation sweep

In [ ]:
dilation_configs=[
    dict(depth=3,base_filters=32,kernel_size=3,pool="max",
         pool_kernel=2,pool_stride=2,dropout=0.3,batchnorm=True,
         dilation=d,channel_growth=2,max_channels=256)
    for d in [1,2,3]
]
dilation_results=controlled_cnn_experiment(dilation_configs)
display(dilation_results)

### 13B. Depth experiment

In [ ]:
depth_configs = [
    dict(depth=d, base_filters=32, kernel_size=3, pool="max", dropout=0.3, batchnorm=True)
    for d in [2,3,4]
]
depth_results = controlled_cnn_experiment(depth_configs)
display(depth_results.sort_values("val_roc_auc", ascending=False))

### 13C. Dropout experiment

In [ ]:
dropout_configs = [
    dict(depth=3, base_filters=32, kernel_size=3, pool="max", dropout=d, batchnorm=True)
    for d in [0.0, 0.2, 0.5]
]
dropout_results = controlled_cnn_experiment(dropout_configs)
display(dropout_results.sort_values("val_roc_auc", ascending=False))

### 13D. Resolution experiment

Compare 64 / 96 / 128 / 160 / 224.  
The transform never enlarges a smaller original image; it pads instead.

For each size, record:
- validation/test metrics
- training time
- GPU memory
- parameter count (unchanged for this GAP-based CNN)

In [ ]:
resolution_results = []
for size in [64, 96, 128, 160, 224]:
    tr, va, _ = make_loaders(size=size, augment=True)
    if FAST_MODE:
        tr = maybe_subset_loader(tr, 4000)
    model = ConfigurableCNN(depth=3, base_filters=32, kernel_size=3,
                            pool="max", dropout=0.3, batchnorm=True).to(DEVICE)
    t0 = time.time()
    hist = fit_model(model, tr, va, epochs=(1 if FAST_MODE else 6), lr=1e-3)
    elapsed = time.time() - t0
    _, y, p, _ = run_epoch(model, va)
    m = binary_metrics(y, p)
    resolution_results.append({
        "resolution": size,
        "val_accuracy": m["accuracy"],
        "val_roc_auc": m["roc_auc"],
        "val_pr_auc": m["pr_auc_avg_precision"],
        "seconds": elapsed,
        "params": count_params(model),
    })
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

resolution_results = pd.DataFrame(resolution_results)
display(resolution_results)

## 14. Student experiment matrix

Do not run every combination blindly. Assign different groups different research questions.

Suggested variables:
- filters / width: base 16, 32, 64
- pooling: max vs average
- optimizer: SGD, Adam, AdamW
- learning rate: 1e-2, 1e-3, 1e-4
- batch size: 16, 32, 64
- augmentation: none, mild, strong
- batch normalization: on/off
- weight decay: 0, 1e-4, 1e-3

Each group should state a **hypothesis before training**.

# Part D — Modern deep learning architectures

In [ ]:
def make_torchvision_binary_model(name, pretrained=False):
    name = name.lower()

    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        m = models.resnet18(weights=weights)
        m.fc = nn.Linear(m.fc.in_features, 1)

    elif name == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        m = models.mobilenet_v3_small(weights=weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 1)

    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b0(weights=weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 1)

    elif name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        m = models.convnext_tiny(weights=weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 1)

    else:
        raise ValueError("Unknown model")

    return m, weights

for name in ["resnet18", "mobilenet_v3_small", "efficientnet_b0", "convnext_tiny"]:
    m, _ = make_torchvision_binary_model(name, pretrained=False)
    print(name, f"{count_params(m)/1e6:.2f}M params")

## 15. From-scratch architecture comparison

Training all modern architectures from scratch on full data can take substantial GPU time.  
Use one or two models in class; distribute models across student groups.

In [ ]:
ARCHS_TO_RUN = ["resnet18"] if FAST_MODE else [
    "resnet18", "mobilenet_v3_small", "efficientnet_b0", "convnext_tiny"
]

scratch_results = []
tr, va, _ = make_loaders(size=IMG_SIZE, augment=True)
if FAST_MODE:
    tr = maybe_subset_loader(tr, 5000)

for name in ARCHS_TO_RUN:
    print("\nFROM SCRATCH:", name)
    m, _ = make_torchvision_binary_model(name, pretrained=False)
    m = m.to(DEVICE)
    hist = fit_model(m, tr, va, epochs=(1 if FAST_MODE else 8), lr=1e-3)
    _, y, p, _ = run_epoch(m, va)
    met = binary_metrics(y, p)
    scratch_results.append({
        "model": name, "pretrained": False,
        "params": count_params(m),
        "val_accuracy": met["accuracy"],
        "val_roc_auc": met["roc_auc"],
        "val_pr_auc": met["pr_auc_avg_precision"],
        "val_mcc": met["mcc"],
    })
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

scratch_results = pd.DataFrame(scratch_results)
display(scratch_results)

# Part E — Pretrained models and transfer learning

## 16. Feature extraction: freeze the pretrained backbone

TorchVision's `weights=...DEFAULT` loads ImageNet pretrained weights.

Protocol:
1. Load pretrained weights.
2. Freeze all layers.
3. Replace the final classifier and train it.
4. Compare with from-scratch training.

**Research note:** official weight-specific preprocessing is the reference protocol for pretrained models. This course notebook keeps a unified normalization and no-upscale policy to support controlled comparison. For a publication-quality benchmark, use each weight object's official `weights.transforms()` and report it explicitly.

In [ ]:
def freeze_all_except_head(model, name):
    for p in model.parameters():
        p.requires_grad = False

    name = name.lower()
    if name == "resnet18":
        for p in model.fc.parameters():
            p.requires_grad = True
    elif name in ["mobilenet_v3_small", "efficientnet_b0", "convnext_tiny"]:
        for p in model.classifier.parameters():
            p.requires_grad = True
    else:
        raise ValueError(name)
    return model

PRETRAINED_ARCHS = ["resnet18", "efficientnet_b0"] if FAST_MODE else [
    "resnet18", "mobilenet_v3_small", "efficientnet_b0", "convnext_tiny"
]

pretrained_results = []
tr, va, _ = make_loaders(size=224, augment=True)
if FAST_MODE:
    tr = maybe_subset_loader(tr, 5000)

for name in PRETRAINED_ARCHS:
    print("\nPRETRAINED FEATURE EXTRACTOR:", name)
    m, weights = make_torchvision_binary_model(name, pretrained=True)
    m = freeze_all_except_head(m, name).to(DEVICE)
    print("Trainable params:", count_params(m, trainable_only=True))
    hist = fit_model(m, tr, va, epochs=PRETRAIN_EPOCHS, lr=1e-3)
    _, y, p, _ = run_epoch(m, va)
    met = binary_metrics(y, p)
    pretrained_results.append({
        "model": name, "strategy": "frozen_backbone",
        "all_params": count_params(m),
        "trainable_params": count_params(m, trainable_only=True),
        "val_accuracy": met["accuracy"],
        "val_roc_auc": met["roc_auc"],
        "val_pr_auc": met["pr_auc_avg_precision"],
        "val_mcc": met["mcc"],
    })
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pretrained_results = pd.DataFrame(pretrained_results)
display(pretrained_results)

## 17. Fine-tuning

After the new classifier has adapted, unfreeze the network and continue with a **smaller learning rate**.

For time reasons, the example below uses ResNet18.  
Students can repeat the same strategy for EfficientNet or ConvNeXt.

In [ ]:
ft_model, _ = make_torchvision_binary_model("resnet18", pretrained=True)
ft_model = freeze_all_except_head(ft_model, "resnet18").to(DEVICE)

# Stage 1: train only head
_ = fit_model(ft_model, tr, va, epochs=(1 if FAST_MODE else 3), lr=1e-3)

# Stage 2: unfreeze all, smaller LR
for p in ft_model.parameters():
    p.requires_grad = True

ft_history = fit_model(
    ft_model, tr, va,
    epochs=(1 if FAST_MODE else 4),
    lr=1e-4, optimizer_name="adamw", weight_decay=1e-4
)

_, y, p, _ = run_epoch(ft_model, va)
display(pd.DataFrame([binary_metrics(y, p)]))

# Part F — Error analysis

## 18. Find high-confidence mistakes

A model with high overall accuracy can still fail in systematic ways.

In [ ]:
# Evaluate current fine-tuned model on independent test
_, y_true, probs, paths = run_epoch(ft_model, test_loader)
preds = (probs >= 0.5).astype(int)

errors = pd.DataFrame({
    "path": paths, "y_true": y_true.astype(int),
    "prob_dog": probs, "pred": preds
})
errors = errors[errors["y_true"] != errors["pred"]].copy()
errors["confidence"] = np.where(errors["pred"]==1, errors["prob_dog"], 1-errors["prob_dog"])
errors = errors.sort_values("confidence", ascending=False)
display(errors.head(20))

In [ ]:
def show_error_cases(error_df, n=12):
    sample = error_df.head(n)
    cols = 4
    rows = max(1, math.ceil(len(sample)/cols))
    plt.figure(figsize=(12, 3*rows))
    for i, (_, r) in enumerate(sample.iterrows(), 1):
        with Image.open(r["path"]) as im:
            plt.subplot(rows, cols, i)
            plt.imshow(im.convert("RGB"))
            plt.axis("off")
            truth = "dog" if r["y_true"] else "cat"
            pred = "dog" if r["pred"] else "cat"
            plt.title(f"T:{truth} P:{pred}\np(dog)={r['prob_dog']:.3f}")
    plt.tight_layout()
    plt.show()

show_error_cases(errors, 12)

# Part G — Hard / OOD Challenge Benchmark

## 19. Challenge-set folder format

Prepare a ZIP with this structure:

```text
challenge_set/
  cat/
    hard_cat_001.jpg
    ...
  dog/
    hard_dog_001.jpg
    ...
  ood/
    fox_001.jpg
    wolf_001.jpg
    stuffed_animal_001.jpg
    ...
```

### Minimum student requirement
- 20+ hard cats
- 20+ hard dogs
- 10+ OOD images

Recommended hard cases: occlusion, extreme scale, blur, low light, unusual viewpoint, rare breed, misleading background, cat+dog mixed scene.

**Important:** OOD images do not have a correct cat/dog label. Measure confidence behavior, not binary accuracy.

In [ ]:
# Upload a challenge_set.zip from your computer in Colab.
# Uncomment when ready:
#
# from google.colab import files
# uploaded = files.upload()
# !unzip -q challenge_set.zip -d /content/challenge

CHALLENGE_ROOT = Path("/content/challenge/challenge_set")
print("Expected challenge root:", CHALLENGE_ROOT)

In [ ]:
def predict_folder(model, folder, size=224):
    tfm = make_transform(size=size, train=False)
    rows = []
    folder = Path(folder)
    if not folder.exists():
        return pd.DataFrame()

    for p in sorted(folder.rglob("*")):
        if not p.is_file():
            continue
        try:
            with Image.open(p) as im:
                x = tfm(im.convert("RGB")).unsqueeze(0).to(DEVICE)
            model.eval()
            with torch.no_grad():
                prob = torch.sigmoid(model(x)).item()
            rows.append({
                "path": str(p),
                "prob_dog": prob,
                "predicted": "dog" if prob >= 0.5 else "cat",
                "binary_confidence": max(prob, 1-prob),
            })
        except Exception:
            pass
    return pd.DataFrame(rows)

# Example after upload:
# hard_cat = predict_folder(ft_model, CHALLENGE_ROOT/"cat", 224)
# hard_dog = predict_folder(ft_model, CHALLENGE_ROOT/"dog", 224)
# ood = predict_folder(ft_model, CHALLENGE_ROOT/"ood", 224)
# display(hard_cat.head(), hard_dog.head(), ood.sort_values("binary_confidence", ascending=False).head(20))

## 20. Suggested challenge analysis

For hard cat/dog images:
- Accuracy
- Sensitivity / Specificity
- F1
- ROC-AUC / PR-AUC if enough labeled cases

For OOD:
- distribution of binary confidence
- number of OOD images with confidence > 0.9
- examples of confidently wrong / inappropriate closed-set predictions

Discussion question:

> Why can a binary classifier confidently call a fox a dog even though "fox" never appeared in training?

This introduces **closed-set vs. open-set recognition**, uncertainty, and distribution shift.

# Part H — Save results for the academic report

In [ ]:
# Save important tables
history.to_csv(RESULTS_DIR / "baseline_history.csv", index=False)
kernel_results.to_csv(RESULTS_DIR / "kernel_results.csv", index=False)
depth_results.to_csv(RESULTS_DIR / "depth_results.csv", index=False)
dropout_results.to_csv(RESULTS_DIR / "dropout_results.csv", index=False)
resolution_results.to_csv(RESULTS_DIR / "resolution_results.csv", index=False)
scratch_results.to_csv(RESULTS_DIR / "scratch_architecture_results.csv", index=False)
pretrained_results.to_csv(RESULTS_DIR / "pretrained_results.csv", index=False)

torch.save(ft_model.state_dict(), RESULTS_DIR / "resnet18_finetuned_state_dict.pt")

print("Saved to:", RESULTS_DIR)
print(list(RESULTS_DIR.iterdir()))

# Part I — Required student report

## Report checklist

### Dataset
- How many valid images?
- How many corrupted images?
- Class balance?
- Original resolution distribution?
- Exact train/validation/test split?

### Baseline CNN
- Architecture diagram/table
- Kernel sizes, channels, pooling, dropout
- Parameter count
- Training settings

### Controlled experiments
At least **four** of:
- depth
- kernel size
- width / filters
- pooling
- dropout
- input resolution
- batch size
- optimizer
- learning rate
- augmentation

### Modern architectures
Compare at least **three** architectures.

### Transfer learning
Compare:
1. from scratch
2. pretrained frozen backbone
3. pretrained + fine-tuning

### Evaluation
Report:
- Accuracy
- Balanced Accuracy
- Precision
- Sensitivity / Recall
- Specificity
- F1
- ROC-AUC
- PR-AUC
- MCC
- TP / TN / FP / FN
- training time
- parameter count

### Independent challenge
Compare standard test performance with hard-image performance, and analyze OOD confidence.

### Scientific conclusion
Do not write only "Model X had the highest accuracy."  
Explain **why the result may have occurred, what the evidence supports, and what limitations remain**.

# Part J — Bridge to Computational Biology

The visual CNN experiment is preparation for the next module:

```text
Image                         Protein
-----                         -------
pixel / patch                 amino acid / local window
2D CNN                        1D CNN
visual feature                sequence motif
ImageNet pretraining          protein-sequence pretraining
pretrained CNN                ESM / ProtT5
hard/OOD image                difficult / remote-homology sequence
image leakage                 sequence homology leakage
```

Next research pipeline:

```text
Protein sequence
    ↓
Protein Language Model
    ↓
Residue embeddings (L × D)
    ↓
1D CNN / multi-window CNN
    ↓
Pooling
    ↓
Biological prediction
```

The same questions return:
- How should we split data?
- What is a fair independent test?
- Which local window/kernel sizes matter?
- Does pretraining improve generalization?
- Does the model still work on genuinely difficult unseen examples?

# Public CNN courses used as design references

- Hung-yi Lee, Machine Learning 2021: receptive field, parameter sharing, filters/feature maps, pooling, CNN limitations.
- Stanford CS231n: sizing formulas, stride/padding, receptive field, small-filter and pooling rules.
- DeepLearning.AI CNN: edge detection, padding, strided convolution, volumes, pooling, why convolutions, case studies.
- PyTorch official transfer learning: frozen feature extractor and fine-tuning.

This notebook extends those teaching ideas into a graduate-level controlled experimental matrix.